In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import minimize

#Carga de datos de valoraciones
data = sio.loadmat('ex8_movies.mat')
Y, R = data['Y'], data['R']

#Estadísticas básicas
n_movies, n_users = Y.shape
avg_toy_story = np.mean(Y[0, R[0, :] == 1])

print(f"Dataset: {n_movies} películas y {n_users} usuarios.")
print(f"Rating promedio de Toy Story: {avg_toy_story:.2f}")

#Carga de parámetros pre-entrenados para validación inicial
params_data = sio.loadmat('ex8_movieParams.mat')
X = params_data['X']
Theta = params_data['Theta']

Dataset: 1682 películas y 943 usuarios.
Rating promedio de Toy Story: 3.88


### Implementamos funcion coste y gradiente

In [2]:
def collaborative_cost_gradient(params, Y, R, n_features, reg_lambda):
    
    #Calculamos el coste y el gradiente para el filtrado colaborativo.
    
    n_movies, n_users = Y.shape
    
    #Hacemos unroll de matrices X y Theta
    X = params[:n_movies * n_features].reshape(n_movies, n_features)
    Theta = params[n_movies * n_features:].reshape(n_users, n_features)
    
    #Error de predicción (solo donde R == 1)
    error = (X @ Theta.T - Y) * R
    
    #Coste con regularización
    cost = 0.5 * np.sum(error**2)
    reg_term = (reg_lambda / 2) * (np.sum(X**2) + np.sum(Theta**2))
    total_cost = cost + reg_term
    
    #Gradientes
    X_grad = (error @ Theta) + (reg_lambda * X)
    Theta_grad = (error.T @ X) + (reg_lambda * Theta)
    
    #Aplanar gradientes para el optimizador
    grad = np.concatenate([X_grad.ravel(), Theta_grad.ravel()])
    
    return total_cost, grad

### Entrenamos el modelo

In [3]:
#Definimos hiperparámetros
N_FEATURES = 10
LAMBDA = 1.5

#Inicializamos valores de forma aleatoria
X_init = np.random.randn(n_movies, N_FEATURES) * 0.1
Theta_init = np.random.randn(n_users, N_FEATURES) * 0.1
initial_params = np.concatenate([X_init.ravel(), Theta_init.ravel()])

#Optimizamos mediante el uso del gradiente conjugado

res = minimize(fun=collaborative_cost_gradient,
               x0=initial_params,
               args=(Y, R, N_FEATURES, LAMBDA),
               method='CG',
               jac=True,
               options={'maxiter': 200, 'disp': True})

#Obtenemos las matrices optimizadas
trained_params = res.x
X_final = trained_params[:n_movies * N_FEATURES].reshape(n_movies, N_FEATURES)
Theta_final = trained_params[n_movies * N_FEATURES:].reshape(n_users, N_FEATURES)

         Current function value: 33084.465683
         Iterations: 200
         Function evaluations: 309
         Gradient evaluations: 309


C:\Users\varea.LAPTOP-P3D215RM\anaconda3\envs\entornoIA2425\lib\site-packages\scipy\optimize\_minimize.py:706: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_cg(fun, x0, args, jac, callback, **options)


### Generamos las recomendaciones

In [4]:
#Generamos la matriz de todas las predicciones
all_predictions = X_final @ Theta_final.T

#Elegimos un usuario en concreto, por ejemplo el que tiene id = 200
user_id = 200
user_preds = all_predictions[:, user_id]

#Cargamos los nombres de películas
with open('movie_ids.txt', encoding='ISO-8859-1') as f:
    movie_list = [" ".join(line.split()[1:]) for line in f]

#Creamos un dataframe para hacer mas comodo el filtrado y ordenación
results = pd.DataFrame({
    'Title': movie_list,
    'Prediction': user_preds,
    'Watched': R[:, user_id]
})

#Filtramos las películas que no se han visto y mostramos las 10 mejores
top_10 = results[results['Watched'] == 0].sort_values(by='Prediction', ascending=False).head(10)

print("\nRecomendaciones Top 10 para el usuario {user_id}:")
print("-" * 50)
for i, row in top_10.iterrows():
    print(f"Predicción: {row['Prediction']:.2f} | Película: {row['Title']}")


Recomendaciones Top 10 para el usuario {user_id}:
--------------------------------------------------
Predicción: 4.78 | Película: Night of the Living Dead (1968)
Predicción: 4.43 | Película: Sunset Blvd. (1950)
Predicción: 4.42 | Película: Paths of Glory (1957)
Predicción: 4.41 | Película: Paradise Lost: The Child Murders at Robin Hood Hills (1996)
Predicción: 4.38 | Película: Once Upon a Time in the West (1969)
Predicción: 4.37 | Película: Belle de jour (1967)
Predicción: 4.35 | Película: Microcosmos: Le peuple de l'herbe (1996)
Predicción: 4.35 | Película: Bonnie and Clyde (1967)
Predicción: 4.29 | Película: Dead Man (1995)
Predicción: 4.26 | Película: Bride of Frankenstein (1935)
